### Without Agent

In [3]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:20b")

response = model.invoke([{"role": "user", "content": "What's the weather in Amman?"}])

response.content

'I’m sorry, but I don’t have the ability to pull live weather data.  \n\nFor the most up‑to‑date forecast in Amman, you can check:\n\n- **Weather websites** such as [Weather.com](https://weather.com/), [AccuWeather](https://www.accuweather.com/), or the local “Jordan Meteorological Department” site.  \n- **Weather apps** on your phone (Apple Weather, Google Weather, Weather Underground, etc.).  \n- **Search engines**: typing “Amman weather” into Google or Bing will give a quick snapshot plus a 7‑day outlook.\n\nIf you let me know the date you’re interested in, I can give you a typical climate pattern for that time of year!'

### With Agent

In [ ]:
import requests
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama


@tool
def get_weather(city: str) -> str:
    # The docstring below is what the model reads to understand this tool
    """Get the current weather for a city.""" # used by the model to understand what this tool does
    # 1. Find the city's coordinates
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1},
    ).json()
    if not geo.get("results"):
        return f"I couldn't find a city called {city}."
    place = geo["results"][0]

    # 2. Get the current weather there
    weather = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,wind_speed_10m,relative_humidity_2m",
        },
    ).json()["current"]

    return (
        f"{place['name']}, {place.get('country', '')}: "
        f"{weather['temperature_2m']}°C, humidity {weather['relative_humidity_2m']}%, "
        f"wind {weather['wind_speed_10m']} km/h"
    )


model = ChatOllama(model="gpt-oss:20b")
agent = create_agent(model=model, tools=[get_weather])

while True:
    question = input("You (press Enter to stop): ").strip()
    if question.lower() in ("", "exit", "quit"):
        print("Bye!")
        break
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("Agent:", result["messages"][-1].content)

Agent: **Amman, Jordan**  
- Temperature: **18.8 °C**  
- Humidity: **43 %**  
- Wind: **3.1 km/h**
Bye!


In [2]:
import os
import requests
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_ollama import ChatOllama
from dotenv import load_dotenv
load_dotenv()

# Serper.dev API key, used to fetch Google search results as JSON.
# Serper is a third-party service, not Google's official API.
# Get a free key (2,500 queries) at https://serper.dev, then set it before running:
#   export SERPER_API_KEY="your_key"
SERPER_API_KEY = os.environ["SERPER_API_KEY"]


@tool
def google_search(query: str) -> str:
    # The docstring below is what the model reads to understand this tool
    """Search Google for current or recent information, such as weather, news, or prices."""
    response = requests.post(
        "https://google.serper.dev/search",
        headers={"X-API-KEY": SERPER_API_KEY, "Content-Type": "application/json"},
        json={"q": query, "num": 5},
        timeout=15,
    )
    response.raise_for_status()
    data = response.json()

    lines = []

    # 1. Google's direct answer box (often has the weather or a quick fact)
    box = data.get("answerBox")
    if box:
        answer = box.get("answer") or box.get("snippet") or box.get("title")
        if answer:
            lines.append(f"Direct answer: {answer}")

    # 2. The normal search results
    for item in data.get("organic", [])[:5]:
        lines.append(f"- {item.get('title')}: {item.get('snippet')} ({item.get('link')})")

    if not lines:
        return f"No Google results found for: {query}"
    return "\n".join(lines)


model = ChatOllama(model="gpt-oss:20b")
agent = create_agent(model=model, tools=[google_search])

while True:
    question = input("You (press Enter to stop): ").strip()
    if question.lower() in ("", "exit", "quit"):
        print("Bye!")
        break
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    print("Agent:", result["messages"][-1].content)

Agent: **Market reaction to Nvidia’s most recent earnings (released 17 Nov 2026)**  

| What happened | How the market responded |
|---------------|--------------------------|
| **Nvidia’s earnings beat analysts’ expectations** – revenue and earnings per share were higher than forecasts, driven largely by AI‑related GPU sales. | **Shares jumped early in the session.** In the first half‑hour after the call, Nvidia stock was up about **2 %**. The gain was driven by optimism that the company will continue to dominate the AI accelerator market. |
| **The broader market reacted more mutedly.** | The S&P 500 slipped roughly **0.5 %** in regular trading, and the Nasdaq Composite fell around **0.6 %**. Tech‑heavy indices saw a broad‑based pull‑back because investors were wary of over‑valuation and of potential “AI‑bubble” concerns. |
| **After the bell, the upside was largely taken.** | In late‑trading, Nvidia’s shares fell back to near‑pre‑earnings levels, ending the day about **2–3 % lower**